In [1]:
import pandas as pd
import re
import json

In [14]:
# Define the path to the CSV you want to load:
# path = '/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen25_72B_etpr_cog_mci_park.csv'
# path = '/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen3_14B_etpr_cog_mci_park_nocop.csv'
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_cog_status_summary/COG_STATUS_test.csv"
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_neuropath_summary/NP_MAJOR_test.csv"
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv"
path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv"

# Read the data
summaries = pd.read_csv(path)


/scratch/5666463.1.cds-gpu/ipykernel_1165070/2666064292.py:10: DtypeWarning: Columns (20,22,24,28,43,45,50,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,155,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,819,825,892,948,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  summaries = pd.read_csv(path)


In [15]:
len(summaries)

5397

In [16]:
import pandas as pd
import re
from collections import Counter

def find_repeated_sentences_in_text(text):
    # Split into sentences using punctuation (adjust if needed)
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    normalized = [s.strip().lower() for s in sentences if s.strip()]
    counter = Counter(normalized)
    repeated = {sent: count for sent, count in counter.items() if count > 1}
    return repeated

# Load your dataframe
# df = pd.read_csv("your_file.csv")  # Example
# Example dataframe:
# df = pd.DataFrame({'visit_summary': [...texts...]})

repeats_info = []

for idx, row in summaries.iterrows():
    text = row['visit_summary']
    repeats = find_repeated_sentences_in_text(text)
    if repeats:
        for sentence, count in repeats.items():
            repeats_info.append({
                "row_index": idx,
                "sentence": sentence,
                "count": count
            })

# Convert results to a DataFrame for easy analysis
repeats_df = pd.DataFrame(repeats_info)

In [17]:
if 'row_index' in repeats_df:
    repeat_list = set(repeats_df['row_index'])
else:
    repeat_list = set()

In [18]:
repeat_list

set()

In [19]:
summaries_dropped_index = summaries.drop(repeat_list).reset_index(drop=True)

In [20]:
# dropped = summaries.loc[list(repeat_list)]

In [21]:
# summaries.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv", index=False)

In [22]:
# summaries_dropped_index = summaries

In [23]:
ages = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    ages.append(pat['Subject Demographics']["Subject's age at visit"])

In [24]:
index = -1
incorrect_age_count = []
incorrect_cop_age_count = []

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

def remove_incorrect_age(text):
    global count
    global index
    global incorrect_cop_age
    index += 1
    # Process text line by line
    modified_text = []
    for sentence in text.split("."):
        match = re.search(remove_pattern, sentence)
        if match:  # Check if the sentence matches the pattern
            # print(sentence)
            age = match.group(1)  # Extract age
            # if "co-participant" in sentence.lower() or "spouse" in sentence.lower() or "child" in sentence.lower() or "daughter" in sentence.lower() or "friend" in sentence.lower() or "son" in sentence.lower() or "sibling" in sentence.lower() or "paid caregiver" in sentence.lower() or "relative" in sentence.lower():
            #     incorrect_cop_age_count.append((index, age))
                
            # else:   
            #     if int(age) != int(ages[index]):
            #         incorrect_age_count.append((index, age))
                    
            if int(age) != int(ages[index]):
                # print(text)
                incorrect_age_count.append((index, age, match))
            
    
    # return final_text

summaries_dropped_index["visit_summary"].apply(remove_incorrect_age)
print("Total sentences with incorrect co participant age:", len(incorrect_cop_age_count))
print("Total sentences with incorrect age:", len(incorrect_age_count))

Total sentences with incorrect co participant age: 0
Total sentences with incorrect age: 0


In [25]:
incorrect_age_count

[]

In [26]:
smoke_quit_age = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    try:
        smoke_quit_age.append(pat['Subject Health History']["If the subject quit smoking, age at which he/she last smoked (i.e., quit) (range (7 - 110))"])
    except:
        smoke_quit_age.append(-1)

In [27]:
import re

# Define regex pattern to identify relevant sentences
pattern = r"\b\d{1,3}-year-old individual\b.*?\bsmoking\b|\bsmoking\b.*?\b\d{1,3}-year-old\b"

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

# Regex pattern to match "The patient quit smoking at the age of X"
quit_smoking_pattern = r"The patient quit smoking at the age of (\d+)"


index = 0
incorrect_smoking_age_count = []
invalid_smoking_age_count = []
quit_30_days_count = []

def clean_smoking_part(text):
    global count
    global index
    global invalid_smoking_age
    global quit_30_days_count
    
    modified_text = []
    
    for sentence in text.split("."):
        # sentence = sentence.strip()  # Remove leading/trailing spaces
        
        if "They quit smoking 30 days prior to the initial visit" in sentence:
            # sentence = sentence.replace("They quit smoking 30 days prior to the initial visit", "They did not smoke cigarettes for 30 days prior to the visit")
            quit_30_days_count.append(index)
        
        # Check for "The patient quit smoking at the age of X"
        quit_match = re.search(quit_smoking_pattern, sentence)
        if quit_match:
            age = int(quit_match.group(1))
            if int(age) > int(ages[index]):  # Skip this sentence if age < 60
                invalid_smoking_age_count.append(index)
                continue

        # Check for "n-year-old" + "smoking"
        match = re.search(remove_pattern, sentence)
        # if match and re.search(pattern, sentence):  
        if match and "smoking" in sentence.lower(): 
            age = match.group(1)  # Extract age
            if int(age) != int(ages[index]):
                # sentence = re.sub(remove_pattern, "", sentence).strip()   # Replace "n-year-old" with ""
                incorrect_smoking_age_count.append(index)
            
            if int(age) <= int(ages[index]) and (int(age) == int(smoke_quit_age[index]) and smoke_quit_age[index] != -1):
                # new_sentence = f"\n\nThe patient quit smoking at the age of {age}. "  # Create new sentence
                # modified_text.append(new_sentence + sentence)  # Add new sentence before original
                incorrect_smoking_age_count.append(index)
            # else:
            #     modified_text.append("\n\n" + sentence)
                # print(index)
                # raise ValueError
        # else:
        #     modified_text.append(sentence)  # Keep other sentences unchanged

    # Join sentences back into a full text
    # final_text = ".".join(modified_text)  # Removes any accidental empty sentences
    index += 1
    # return final_text

# Apply function to DataFrame column
summaries_dropped_index["visit_summary"].apply(clean_smoking_part)

print("Total sentences modified:", len(incorrect_smoking_age_count))
print("Invalid smoking age:", len(invalid_smoking_age_count))
print("Invalid quit_smoking_days:", len(quit_30_days_count))


Total sentences modified: 0
Invalid smoking age: 0
Invalid quit_smoking_days: 0
